# Can the tiny causal LM overfit one fixed batch?

This notebook checks whether the package's tiny debug causal language model can memorize one fixed token batch. It imports the model from `tiny_stories_experiment` and does not keep a private copy of the network.

## Question and success signal

Can `ModelSpec.tiny_debug()` reach a next-token training loss below 0.05 on one fixed synthetic batch?

Success uses the same threshold as `tests/unit/infrastructure/test_overfit_one_batch.py`: the last training step's loss is below 0.05. That result says the Lightning adapter can memorize a tiny batch on CPU. It does not say the model is trained on TinyStories.

## Setup and configuration

Kernel: the project environment (`uv run`, CPU PyTorch wheel). Working directory: repository root, so `tiny_stories_experiment` imports.

No credentials, raw dataset, or tokenizer file are required. The ROCm container can open this notebook later; this check itself is the CPU overfit path.

In [1]:
from tiny_stories_experiment.domain.modeling.model_spec import ModelSpec
from tiny_stories_experiment.infrastructure.lightning.causal_lm import (
    FIXED_OVERFIT_TOKEN_IDS,
    NEAR_ZERO_TRAINING_LOSS,
    OVERFIT_STEP_COUNT,
    build_tiny_debug_model,
    fit_token_batch,
)

spec = ModelSpec.tiny_debug()
model = build_tiny_debug_model()
parameter_count = sum(parameter.numel() for parameter in model.parameters())
spec, parameter_count, NEAR_ZERO_TRAINING_LOSS, OVERFIT_STEP_COUNT

(ModelSpec(vocab_size=64, embedding_size=64, layer_count=2, attention_head_count=4, context_length=32, feedforward_size=128),
 73216,
 0.05,
 250)

## Inputs and assumptions

The batch is the package constant `FIXED_OVERFIT_TOKEN_IDS`: two short rows of synthetic token ids, reused for every step. It is not text from the prepared TinyStories files.

Assumptions: CPU execution, the tiny debug spec (at most 500_000 parameters), and the adapter's default Adam step count. A low loss means this batch was memorized, not that generation quality is good.

In [2]:
FIXED_OVERFIT_TOKEN_IDS

((1, 2, 3, 4, 5, 6, 7, 8), (8, 7, 6, 5, 4, 3, 2, 1))

## Exploration

Fit the same batch for `OVERFIT_STEP_COUNT` steps through `fit_token_batch`. The returned value is the loss measured on the last step before that step's update.

In [3]:
final_loss = fit_token_batch(model, FIXED_OVERFIT_TOKEN_IDS, OVERFIT_STEP_COUNT)
print(f"final_loss={final_loss}")
assert final_loss < NEAR_ZERO_TRAINING_LOSS
assert NEAR_ZERO_TRAINING_LOSS == 0.05
final_loss

final_loss=0.00023683659674134105


0.00023683659674134105

## Observations and next decision

The cell above prints the final loss and asserts it is below 0.05. When that assertion passes, the overfit check agrees with the CPU pytest.

Still out of scope: pointing `make train` at a real training entrypoint, full-corpus training, and evaluate/generate commands.